# Inter-Subset Analysis

This notebook looks at benchmarks with pre-defined subsets and analyzes the differences between:

1. Model ranking on the full benchmark vs each subset
2. Model performance on the full benchmark vs each subset

Import packages.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from scipy.stats import kendalltau
from tqdm import tqdm
from src.utils.enums import Dataset, MetadataKey
from src.utils.data import (
    get_metadata_mask,
    get_unique_metadata_values,
    load_dataset,
)
from src.utils.model import compute_model_ranking, load_model_scores
from src.utils.path import build_plot_path
from src.utils.plot import plot_histogram

We'll only analyze DS-1000, MATH, and MMLU because we determined that those were the only benchmarks with pre-defined categories and per-instance model scores during EDA.

In [2]:
datasets = [Dataset.MATH, Dataset.MMLU, Dataset.DS_1000]

## DS-1000

Load the dataset and model scores.
- **Note:** We'll categorize instances based on the library that's used (e.g., numpy, pandas, etc.) or the perturbation type

In [3]:
dataset = Dataset.DS_1000
dataset_df = load_dataset(dataset)
model_scores_df = load_model_scores(dataset)
models = model_scores_df.columns.tolist()

print(f"Number of instances: {(num_instances := len(dataset_df))}")
display(dataset_df.head())
print("-" * 200)
print(f"Number of models: {(num_models := len(models))}")
display(model_scores_df.head())

assert len(dataset_df) == len(model_scores_df)

Number of instances: 1000


,prompt,reference_code,metadata,code_context
0,Problem:\nI have the following DataFrame:\n ...,"def g(df, List):\n return df.iloc[List]\n\n...","{'problem_id': 0, 'library_problem_id': 0, 'li...",import pandas as pd\nimport numpy as np\nimpor...
1,Problem:\nI have the following DataFrame:\n ...,"def g(df, List):\n df2 = df.iloc[List].rein...","{'problem_id': 1, 'library_problem_id': 1, 'li...",import pandas as pd\nimport numpy as np\nimpor...
2,Problem:\nI have following pandas dataframe :\...,def g(df):\n return df.where(df.apply(lambd...,"{'problem_id': 2, 'library_problem_id': 2, 'li...",import pandas as pd\nimport numpy as np\nimpor...
3,Problem:\nI have following pandas dataframe :\...,def g(df):\n return df.where(df.apply(lambd...,"{'problem_id': 3, 'library_problem_id': 3, 'li...",import pandas as pd\nimport numpy as np\nimpor...
4,Problem:\nI have following pandas dataframe :\...,result = df.where(df.apply(lambda x: x.map...,"{'problem_id': 4, 'library_problem_id': 4, 'li...",import pandas as pd\nimport numpy as np\nimpor...


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Number of models: 3


,deepseek-coder-6.7b-base,gpt-3.5-turbo-0613,gpt-4o-2024-08-06
0,1,0,0
1,0,1,1
2,1,0,1
3,1,1,1
4,0,1,0


### Ranking Analysis

Compute the global model ranking.

In [4]:
global_model_to_rank = compute_model_ranking(model_scores_df)

for model, rank in global_model_to_rank.items():
    print(f"{model}: {rank}")

global_ranking = list(global_model_to_rank.values())

deepseek-coder-6.7b-base: 3.0
gpt-3.5-turbo-0613: 2.0
gpt-4o-2024-08-06: 1.0


Count the number of subsets where the local model ranking differs from the global model ranking.

In [5]:
subset_col = MetadataKey.LIBRARY
subsets = get_unique_metadata_values(dataset_df, subset_col)
print(f"Subsets: {subsets}")

num_different = 0

for subset in subsets:
    mask = get_metadata_mask(dataset_df, subset_col, subset)
    subset_model_scores = model_scores_df[mask]

    subset_model_to_rank = compute_model_ranking(subset_model_scores)
    subset_ranking = list(subset_model_to_rank.values())

    num_different += 1 if not np.array_equal(subset_ranking, global_ranking) else 0

print(f"Total number of subsets: {(num_subsets := len(subsets))}")
print(f"Subsets that differ from global ranking: {num_different}")

Subsets: {'Pytorch', 'Tensorflow', 'Pandas', 'Matplotlib', 'Numpy', 'Sklearn', 'Scipy'}
Total number of subsets: 7
Subsets that differ from global ranking: 1


Compute the Kendall's Tau between each subset's local model ranking and the global model ranking. 

In [6]:
kwargs = {
    "desc": "Computing model rankings",
    "total": len(subsets),
    "unit": "subset",
}

kendall_tau_results = []

for subset in tqdm(subsets, **kwargs):
    mask = get_metadata_mask(dataset_df, subset_col, subset)
    subset_model_scores = model_scores_df[mask]

    subset_model_to_rank = compute_model_ranking(subset_model_scores)
    subset_ranking = list(subset_model_to_rank.values())

    kendall_tau, _ = kendalltau(global_ranking, subset_ranking)
    kendall_tau_results.append(
        {
            "subset": subset,
            "ranking": subset_ranking,
            "kendall_tau": kendall_tau,
            "n_instances": len(subset_model_scores),
        }
    )

kendall_tau_df = pd.DataFrame(kendall_tau_results)
kendall_tau_df.head()

Computing model rankings: 100%|██████████| 7/7 [00:00<00:00, 693.31subset/s]


,subset,ranking,kendall_tau,n_instances
0,Pytorch,"[3.0, 2.0, 1.0]",1.0,68
1,Tensorflow,"[3.0, 2.0, 1.0]",1.0,45
2,Pandas,"[3.0, 2.0, 1.0]",1.0,291
3,Matplotlib,"[3.0, 2.0, 1.0]",1.0,155
4,Numpy,"[3.0, 2.0, 1.0]",1.0,220


Plot the distribution of Kendall's Taus between each subset's local model ranking and the global model ranking.

In [ ]:
column = "kendall_tau"
xlabel = "Kendall's Tau"
ylabel = f"{subset_col.capitalize()} Count"
title = (
    f"{dataset}: Kendall's Tau Across Libraries"
    f"\n({num_models} models, {len(subsets)} subsets, {num_instances} instances)"
)
annotate = True
mean = kendall_tau_df[column].mean()
std = kendall_tau_df[column].std()
xlim = (0, 1)
ylim = (0, num_subsets)

fig = plot_histogram(
    kendall_tau_df[column],
    xlabel=xlabel,
    ylabel=ylabel,
    title=title,
    annotate=annotate,
    mean=mean,
    std=std,
    xlim=xlim,
)

analysis = "inter_subset_analysis"
plot_name = f"kendall_tau_distribution_{subset_col}"
file_path = build_plot_path(dataset, analysis, plot_name)
plt.savefig(file_path)
plt.show()

### Performance Analysis

Compute each model's mean performance across the entire benchmark.

In [8]:
mean_accuracy_df = (
    model_scores_df.mean(axis=0)
    .reset_index(name="accuracy")
    .rename(columns={"index": "model"})
)
mean_accuracy_df.sort_values(by="accuracy", ascending=False, inplace=True)
mean_accuracy_df.round(3).head()

,model,accuracy
2,gpt-4o-2024-08-06,0.598
1,gpt-3.5-turbo-0613,0.387
0,deepseek-coder-6.7b-base,0.312


Compute each model's mean performance on each subset.

In [9]:
kwargs = {
    "desc": "Computing mean performance",
    "total": len(subsets),
    "unit": "subset",
}

accuracy_results = {}

for subset in tqdm(subsets, **kwargs):
    mask = get_metadata_mask(dataset_df, subset_col, subset)
    subset_model_scores = model_scores_df[mask]
    accuracy_results[subset] = subset_model_scores.mean()

accuracy_df = pd.DataFrame(accuracy_results).T
accuracy_df.index.name = "subset"
accuracy_df.head()

Computing mean performance: 100%|██████████| 7/7 [00:00<00:00, 1600.53subset/s]


,deepseek-coder-6.7b-base,gpt-3.5-turbo-0613,gpt-4o-2024-08-06
subset,,,
Pytorch,0.191176,0.294118,0.676471
Tensorflow,0.222222,0.333333,0.488889
Pandas,0.206186,0.329897,0.463918
Matplotlib,0.522581,0.593548,0.754839
Numpy,0.354545,0.368182,0.700000


Plot the distribution of each model's mean performance across the different subsets.

In [ ]:
global_means = model_scores_df.mean()
fig, axes = plt.subplots(num_models, 1, figsize=(8, 3 * num_models))

for i, model in enumerate(accuracy_df.columns):
    column = model
    xlabel = "Mean Accuracy"
    ylabel = f"{subset_col.capitalize()} Count"
    title = f"{model}"
    ax = axes[i] if num_models > 1 else axes
    annotate = True
    mean = global_means[model]
    mean_label = "Benchmark-level accuracy"
    xlim = (0, 1)
    ylim = (0, num_subsets)

    plot_histogram(
        accuracy_df[column],
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        ax=ax,
        annotate=annotate,
        mean=mean,
        mean_label=mean_label,
        xlim=xlim,
        ylim=ylim,
    )

plt.suptitle(
    f"{dataset}: Mean Accuracy Across Libraries"
    f"\n({len(subsets)} libraries, {num_instances} instances)"
)
plt.tight_layout()
plot_name = f"accuracy_distribution_{subset_col}"
file_path = build_plot_path(dataset, analysis, plot_name)
plt.savefig(file_path)
plt.show()

## MATH

Load the dataset and model scores.

In [11]:
dataset = Dataset.MATH
dataset_df = load_dataset(dataset)
model_scores_df = load_model_scores(dataset)
# model_scores_df = model_scores_df.drop(columns=["gpt-4o-mini-2024-07-18"])
models = model_scores_df.columns.tolist()

print(f"Number of instances: {(num_instances := len(dataset_df))}")
display(dataset_df.head())
print("-" * 200)
print(f"Number of models: {(num_models := len(models))}")
display(model_scores_df.head())

assert len(dataset_df) == len(model_scores_df)

Number of instances: 5000


,problem,level,type,solution,subset
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,algebra
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,algebra
2,Find $x$ such that $\lceil x \rceil + x = \dfr...,Level 4,Algebra,"First, we note that $x$ must be positive, sinc...",algebra
3,Evaluate $i^5+i^{-25}+i^{45}$.,Level 5,Algebra,We have $i^5 = i^4\cdot i = 1\cdot (i) = i$. ...,algebra
4,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,algebra


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Number of models: 5


,dart-math-llama3-8b-uniform,deepseek-math-7b-instruct,gpt-4o-mini-2024-07-18,Llama-3-8B-Instruct,Llama-3.1-8B-Instruct
0,1,1,1,1,1
1,1,1,1,0,1
2,0,1,1,1,0
3,1,0,1,0,1
4,1,1,1,1,1


### Ranking Analysis

Compute the global model ranking.

In [12]:
global_model_to_rank = compute_model_ranking(model_scores_df)

for model, rank in global_model_to_rank.items():
    print(f"{model}: {rank}")

global_ranking = list(global_model_to_rank.values())

Llama-3-8B-Instruct: 5.0
Llama-3.1-8B-Instruct: 2.0
dart-math-llama3-8b-uniform: 4.0
deepseek-math-7b-instruct: 3.0
gpt-4o-mini-2024-07-18: 1.0


Count the number of subsets where the local model ranking differs from the global model ranking.

In [13]:
subset_col = "subset"
subsets = dataset_df[subset_col].unique().tolist()
print(f"Subsets: {subsets}")

num_different = 0

for subset in subsets:
    mask = dataset_df[subset_col] == subset
    subset_model_scores = model_scores_df[mask]
    subset_model_to_rank = compute_model_ranking(subset_model_scores)
    subset_ranking = list(subset_model_to_rank.values())
    num_different += 1 if not np.array_equal(subset_ranking, global_ranking) else 0

print(f"Total number of subsets: {(num_subsets:=len(subsets))}")
print(f"Subsets that differ from the global ranking: {num_different}")

Subsets: ['algebra', 'counting_and_probability', 'geometry', 'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus']
Total number of subsets: 7
Subsets that differ from the global ranking: 4


Compute the Kendall's Tau between each subset's model ranking and the global model ranking. 

In [14]:
kwargs = {
    "desc": "Computing model rankings",
    "total": len(subsets),
    "unit": "subset",
}

kendall_tau_results = []

for subset in tqdm(subsets, **kwargs):
    mask = dataset_df[subset_col] == subset
    subset_model_scores = model_scores_df[mask]

    subset_model_to_rank = compute_model_ranking(subset_model_scores)
    subset_ranking = list(subset_model_to_rank.values())

    kendall_tau, _ = kendalltau(global_ranking, subset_ranking)
    kendall_tau_results.append(
        {
            "subset": subset,
            "ranking": subset_ranking,
            "kendall_tau": kendall_tau,
            "n_instances": mask.sum(),
        }
    )

kendall_tau_df = pd.DataFrame(kendall_tau_results)
kendall_tau_df.head()

Computing model rankings: 100%|██████████| 7/7 [00:00<00:00, 978.64subset/s]


,subset,ranking,kendall_tau,n_instances
0,algebra,"[5.0, 2.0, 4.0, 3.0, 1.0]",1.0,1187
1,counting_and_probability,"[5.0, 2.0, 4.0, 3.0, 1.0]",1.0,474
2,geometry,"[5.0, 3.0, 4.0, 2.0, 1.0]",0.8,479
3,intermediate_algebra,"[5.0, 2.0, 3.0, 4.0, 1.0]",0.8,903
4,number_theory,"[5.0, 2.0, 3.0, 4.0, 1.0]",0.8,540


Plot the distribution of Kendall's Taus between each subset's model ranking and the global model ranking.

In [ ]:
column = "kendall_tau"
xlabel = "Kendall's Tau"
ylabel = f"{subset_col.capitalize()} Count"

without_gpt_4o_mini = "gpt-4o-mini-2024-07-18" not in models
note = " (without gpt-4o-mini-2024-07-18)" if without_gpt_4o_mini else ""
title = (
    f"{dataset}: Distribution of Kendall's Taus Across {subset_col.capitalize()}s{note}"
    f"\n({num_models} models, {len(subsets)} subsets, {num_instances} instances)"
)
annotate = True
mean = kendall_tau_df[column].mean()
std = kendall_tau_df[column].std()
xlim = (0, 1)
ylim = (0, num_subsets)

fig = plot_histogram(
    kendall_tau_df[column],
    xlabel=xlabel,
    ylabel=ylabel,
    title=title,
    annotate=annotate,
    mean=mean,
    std=std,
    xlim=xlim,
    ylim=ylim,
)

suffix = "_without_gpt-4o-mini-2024-07-18" if without_gpt_4o_mini else ""
plot_name = f"kendall_tau_distribution{suffix}"
file_path = build_plot_path(dataset, analysis, plot_name)
plt.savefig(file_path)
plt.show()

### Performance Analysis

Compute each model's mean performance across the entire benchmark.

In [16]:
mean_accuracy_df = (
    model_scores_df.mean(axis=0)
    .reset_index(name="accuracy")
    .rename(columns={"index": "model"})
)
mean_accuracy_df.sort_values(by="accuracy", ascending=False, inplace=True)
mean_accuracy_df.round(3).head()

,model,accuracy
2,gpt-4o-mini-2024-07-18,0.701
4,Llama-3.1-8B-Instruct,0.497
1,deepseek-math-7b-instruct,0.470
0,dart-math-llama3-8b-uniform,0.453
3,Llama-3-8B-Instruct,0.301


Compute each model's mean performance on each subset.

In [17]:
kwargs = {
    "desc": "Computing mean performance",
    "total": len(subsets),
    "unit": "subset",
}

accuracy_results = {}

for subset in tqdm(subsets, **kwargs):
    mask = dataset_df[subset_col] == subset
    subset_model_scores = model_scores_df[mask]
    accuracy_results[subset] = subset_model_scores.mean()

accuracy_df = pd.DataFrame(accuracy_results).T
accuracy_df.index.name = "subset"
accuracy_df.head()

Computing mean performance: 100%|██████████| 7/7 [00:00<00:00, 2333.87subset/s]


,dart-math-llama3-8b-uniform,deepseek-math-7b-instruct,gpt-4o-mini-2024-07-18,Llama-3-8B-Instruct,Llama-3.1-8B-Instruct
subset,,,,,
algebra,0.665543,0.691660,0.882056,0.444819,0.717776
counting_and_probability,0.345992,0.375527,0.725738,0.223629,0.426160
geometry,0.323591,0.382046,0.565762,0.208768,0.377871
intermediate_algebra,0.253599,0.225914,0.460687,0.130676,0.260244
number_theory,0.357407,0.353704,0.725926,0.231481,0.444444


Plot the distribution of each model's mean performance across the different subsets.

In [ ]:
global_means = model_scores_df.mean()
fig, axes = plt.subplots(num_models, 1, figsize=(8, 3 * num_models))

for i, model in enumerate(accuracy_df.columns):
    column = model
    xlabel = "Mean Accuracy"
    ylabel = f"{subset_col.capitalize()} Count"
    title = f"{model}"
    ax = axes[i] if num_models > 1 else axes
    annotate = True
    mean = global_means[model]
    mean_label = "Benchmark-level accuracy"
    xlim = (0, 1)
    ylim = (0, num_subsets)

    plot_histogram(
        accuracy_df[column],
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        ax=ax,
        annotate=annotate,
        mean=mean,
        mean_label=mean_label,
        xlim=xlim,
        ylim=ylim,
    )

plt.suptitle(
    f"{dataset}: Mean Accuracy Across {subset_col.capitalize()}s"
    f"\n({len(subsets)} {subset_col.capitalize()}s, {num_instances} instances)"
)
plt.tight_layout()
plot_name = f"accuracy_distribution{suffix}"
file_path = build_plot_path(dataset, analysis, plot_name)
plt.savefig(file_path)
plt.show()

## MMLU

Load the dataset and model scores.

In [19]:
dataset = Dataset.MMLU
dataset_df = load_dataset(dataset)
model_scores_df = load_model_scores(dataset)
models = model_scores_df.columns.tolist()

print(f"Number of instances: {(num_instances := len(dataset_df))}")
display(dataset_df.head())
print("-" * 200)
print(f"Number of models: {(num_models := len(models))}")
display(model_scores_df.head())

assert len(dataset_df) == len(model_scores_df)

Number of instances: 14042


,question,subject,choices,answer
0,Find the degree for the given field extension ...,abstract_algebra,"[0, 4, 2, 6]",1
1,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...",abstract_algebra,"[8, 2, 24, 120]",2
2,Find all zeros in the indicated finite field o...,abstract_algebra,"[0, 1, 0,1, 0,4]",3
3,Statement 1 | A factor group of a non-Abelian ...,abstract_algebra,"[True, True, False, False, True, False, False,...",1
4,Find the product of the given polynomials in t...,abstract_algebra,"[2x^2 + 5, 6x^2 + 4x + 6, 0, x^2 + 1]",1


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Number of models: 9


,claude-3.5-haiku,gpt-3.5-turbo,gpt-4o-mini-2024-07-18,Llama-3.1-70B-Instruct,Llama-3.1-8B-Instruct,Llama-3.1-Tulu-3-70B,Llama-3.1-Tulu-3-8B,Qwen2.5-72B-Instruct,Qwen2.5-7B-Instruct
0,1,0,1,1,0,1,0,1,0
1,0,0,0,1,0,0,0,1,0
2,1,1,0,1,0,0,0,0,1
3,0,0,0,0,0,1,0,1,1
4,1,0,1,1,1,1,0,1,1


### Ranking Analysis

Compute the global model ranking.

In [20]:
global_model_to_rank = compute_model_ranking(model_scores_df)

for model, rank in global_model_to_rank.items():
    print(f"{model}: {rank}")

global_ranking = list(global_model_to_rank.values())

Llama-3.1-70B-Instruct: 1.0
Llama-3.1-8B-Instruct: 7.0
Llama-3.1-Tulu-3-70B: 3.0
Llama-3.1-Tulu-3-8B: 9.0
Qwen2.5-72B-Instruct: 2.0
Qwen2.5-7B-Instruct: 6.0
claude-3.5-haiku: 5.0
gpt-3.5-turbo: 8.0
gpt-4o-mini-2024-07-18: 4.0


Count the number of subsets where the local model ranking differs from the global model ranking.

In [21]:
subset_col = "subject"
subsets = dataset_df[subset_col].unique().tolist()
print(f"Subsets: {subsets}")

num_different = 0

for subset in subsets:
    mask = dataset_df[subset_col] == subset
    subset_model_scores = model_scores_df[mask]
    subset_model_to_rank = compute_model_ranking(subset_model_scores)
    subset_ranking = list(subset_model_to_rank.values())
    num_different += 1 if not np.array_equal(subset_ranking, global_ranking) else 0

print(f"Total number of subsets: {(num_subsets:=len(subsets))}")
print(f"Subsets that differ from the global ranking: {num_different}")

Subsets: ['abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management', 'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios', 'nutrition', 'philosophy'

Compute the Kendall's Tau between each subset's model ranking and the global model ranking. 

In [22]:
kwargs = {
    "desc": "Computing model rankings",
    "total": len(subsets),
    "unit": "subset",
}

kendall_tau_results = []

for subset in tqdm(subsets, **kwargs):
    mask = dataset_df[subset_col] == subset
    subset_model_scores = model_scores_df[mask]

    subset_model_to_rank = compute_model_ranking(subset_model_scores)
    subset_ranking = list(subset_model_to_rank.values())

    kendall_tau, _ = kendalltau(global_ranking, subset_ranking)
    kendall_tau_results.append(
        {
            "subset": subset,
            "ranking": subset_ranking,
            "kendall_tau": kendall_tau,
            "n_instances": mask.sum(),
        }
    )

kendall_tau_df = pd.DataFrame(kendall_tau_results)
kendall_tau_df.head()

Computing model rankings: 100%|██████████| 57/57 [00:00<00:00, 883.53subset/s]


,subset,ranking,kendall_tau,n_instances
0,abstract_algebra,"[2.0, 7.0, 4.5, 9.0, 1.0, 6.0, 4.5, 8.0, 3.0]",0.873326,100
1,anatomy,"[4.0, 7.0, 3.0, 9.0, 2.0, 8.0, 5.0, 6.0, 1.0]",0.555556,135
2,astronomy,"[2.0, 8.0, 1.0, 7.0, 3.0, 6.0, 4.0, 9.0, 5.0]",0.722222,152
3,business_ethics,"[1.0, 8.5, 2.5, 8.5, 2.5, 4.5, 6.0, 7.0, 4.5]",0.841375,100
4,clinical_knowledge,"[1.0, 7.0, 3.0, 9.0, 2.0, 6.0, 5.0, 8.0, 4.0]",1.000000,265


Plot the distribution of Kendall's Taus between each subset's model ranking and the full dataset's model ranking.

In [ ]:
column = "kendall_tau"
xlabel = "Kendall's Tau"
ylabel = f"{subset_col.capitalize()} Count"

without_gpt_4o_mini = "gpt-4o-mini-2024-07-18" not in models
note = " (without gpt-4o-mini-2024-07-18)" if without_gpt_4o_mini else ""
title = (
    f"{dataset}: Distribution of Kendall's Taus Across {subset_col.capitalize()}s{note}"
    f"\n({num_models} models, {len(subsets)} subsets, {num_instances} instances)"
)
annotate = True
mean = kendall_tau_df[column].mean()
std = kendall_tau_df[column].std()
xlim = (0, 1)
ylim = (0, num_subsets // 2)

fig = plot_histogram(
    kendall_tau_df[column],
    xlabel=xlabel,
    ylabel=ylabel,
    title=title,
    annotate=annotate,
    mean=mean,
    std=std,
    xlim=xlim,
    ylim=ylim,
)

suffix = "_without_gpt-4o-mini-2024-07-18" if without_gpt_4o_mini else ""
plot_name = f"kendall_tau_distribution{suffix}"
file_path = build_plot_path(dataset, analysis, plot_name)
plt.savefig(file_path)
plt.show()

### Performance Analysis

Compute each model's mean performance across the entire dataset.

In [24]:
mean_accuracy_df = (
    model_scores_df.mean(axis=0)
    .reset_index(name="accuracy")
    .rename(columns={"index": "model"})
)
mean_accuracy_df.sort_values(by="accuracy", ascending=False, inplace=True)
mean_accuracy_df.round(3).head()

,model,accuracy
3,Llama-3.1-70B-Instruct,0.853
7,Qwen2.5-72B-Instruct,0.850
5,Llama-3.1-Tulu-3-70B,0.819
2,gpt-4o-mini-2024-07-18,0.809
0,claude-3.5-haiku,0.808


Compute each model's mean performance on each subset.

In [25]:
kwargs = {
    "desc": "Computing mean performance",
    "total": len(subsets),
    "unit": "subset",
}

accuracy_results = {}

for subset in tqdm(subsets, **kwargs):
    mask = dataset_df[subset_col] == subset
    subset_model_scores = model_scores_df[mask]
    accuracy_results[subset] = subset_model_scores.mean()

accuracy_df = pd.DataFrame(accuracy_results).T
accuracy_df.index.name = "subset"
accuracy_df.head()

Computing mean performance: 100%|██████████| 57/57 [00:00<00:00, 3460.25subset/s]


,claude-3.5-haiku,gpt-3.5-turbo,gpt-4o-mini-2024-07-18,Llama-3.1-70B-Instruct,Llama-3.1-8B-Instruct,Llama-3.1-Tulu-3-70B,Llama-3.1-Tulu-3-8B,Qwen2.5-72B-Instruct,Qwen2.5-7B-Instruct
subset,,,,,,,,,
abstract_algebra,0.660000,0.410000,0.670000,0.740000,0.430000,0.660000,0.350000,0.780000,0.640000
anatomy,0.770370,0.733333,0.837037,0.800000,0.725926,0.822222,0.696296,0.829630,0.718519
astronomy,0.907895,0.743421,0.888158,0.934211,0.763158,0.940789,0.776316,0.927632,0.835526
business_ethics,0.760000,0.720000,0.780000,0.820000,0.700000,0.800000,0.700000,0.800000,0.780000
clinical_knowledge,0.837736,0.762264,0.849057,0.871698,0.773585,0.856604,0.754717,0.860377,0.815094


Plot the distribution of each model's performance across the different subsets.

In [ ]:
global_means = model_scores_df.mean()
fig, axes = plt.subplots(num_models, 1, figsize=(8, 3 * num_models))

for i, model in enumerate(accuracy_df.columns):
    column = model
    xlabel = "Mean Accuracy"
    ylabel = f"{subset_col.capitalize()} Count"
    title = f"{model}"
    ax = axes[i] if num_models > 1 else axes
    annotate = True
    mean = global_means[model]
    mean_label = "Benchmark-level accuracy"
    xlim = (0, 1)
    ylim = (0, num_subsets // 2)

    plot_histogram(
        accuracy_df[column],
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        ax=ax,
        annotate=annotate,
        mean=mean,
        mean_label=mean_label,
        xlim=xlim,
        ylim=ylim,
    )

plt.suptitle(
    f"{dataset}: Mean Accuracy Across {subset_col.capitalize()}s"
    f"\n({len(subsets)} {subset_col.capitalize()}s, {num_instances} instances)",
    y=1.01,
)
plt.tight_layout()
plot_name = "accuracy_distribution"
file_path = build_plot_path(dataset, analysis, plot_name)
plt.savefig(file_path)
plt.show()